### Library importations 

In [1]:
import os.path as op
import mne 
import os
import numpy as np 
from matplotlib import pyplot as plt
import pandas as pd

### Global variables

In [7]:
baseline_window = 10
window_test = 30 # sliding window over signal 
window = 50 
threshold_test = 100 
r0 = 3 

### Importations 

### Plotting functions 

In [2]:
def on_key(event):
    global current_index, fig, subject_epoch, subject, block,window_size

    if event.key == 'right':
        current_index = (current_index + 1) % len(subject_epoch)
    elif event.key == 'left':
        current_index = (current_index - 1) % len(subject_epoch)
    elif event.key == 'escape':
        print("Quitting the plot!")
        plt.close(fig)
        return

    epoch_zygo = np.squeeze(subject_epoch[current_index].get_data(picks=['Zygo']))
    epoch_corr = np.squeeze(subject_epoch[current_index].get_data(picks=['Corr']))

    epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_fmd_zygo = get_features(epoch_zygo)
    epoch_var_corr, epoch_rms_corr, epoch_wl_corr, epoch_fmd_corr = get_features(epoch_corr)


    plt.close(fig)
    fig = plot_test_variance_window_func(epoch_corr,epoch_zygo,epoch_var_corr,epoch_var_zygo)
  
    fig.canvas.mpl_connect('key_press_event', on_key)   
    fig.suptitle(f"subject: {subject} | block: {block} | epoch: {current_index}", fontsize=14)


    plt.show(block=False)


In [3]:
def plot_test_variance_window_func(corr,zygo,feature_corr,feature_zygo):
    fig, ax = plt.subplots(2, 1, figsize=(8, 4), sharex=True)

    ax_corr = ax[0].twinx()
    ax_zygo = ax[1].twinx()

    title = fig.suptitle(
        "Facial EMG Response Following Stimulus",
        fontsize=14,
    )

    corrugator_color = "#734F62"
    zygomatic_color = "#6F9359"

    time = np.linspace(0,10,2251)
    trigger_time = time[0]

    zygo_out,thr_zyg = test_variance_window_func(feature_zygo)
    zygo_out = zygo_out.astype(int)
    corr_out,thr_corr = test_variance_window_func(feature_corr)
    corr_out = corr_out.astype(int)

                    
    # Plot Corr
    #ax[0].axvline(trigger_time, label="Trigger Channel", color="black")
    ax[0].plot(time,corr,  color=corrugator_color)
   # ax_corr.plot(time,feature_corr*100)
   # ax[0].axhline(thr_corr,linestyle='--')
    ax[0].set_ylim(-200, 200)
    ax[0].set_ylabel("Corr",size=12)
    # ax[0].set_xlim(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))

    # Plot Zygo
    ax[1].plot(time, zygo, lw=1.5, color=zygomatic_color)
   # ax[1].axhline(thr_zyg,linestyle='--')
   # ax_zygo.plot(time,feature_zygo*100)
    ax[1].set_ylim(-200, 200)
    ax[1].set_ylabel("Zygo",size=12)
    # ax[1].set_xlim(int(df_triggers['Time(Sample)'][t]-1),int(df_triggers['Time(Sample)'][t]+inter_trigger_length*frq))


    # plotting features 
    ax_corr.plot(time,corr_out,  color="red", alpha=0.7,)


    ax_zygo.plot(time,zygo_out,  color="red", label="computed labels", alpha=0.7,)

    ax_zygo.axis("on")
    ax_corr.axis("on")

    plt.legend() 
    for a in ax:
        a.spines['top'].set_visible(False)
        a.spines['right'].set_visible(False)

        plt.tight_layout()
        #plt.show()
    return fig 

### Processing functions 

In [4]:
# pre-processes data for each subject and returns df with session divided into 60 epochs w meta-data attached 
def pre_process_subjets(subject,block):
    global raw_path
    global frq

    subject_name = subject + block 
    file= op.join(raw_path,'{}.edf'.format(subject_name))
    raw =  mne.io.read_raw_edf(file,preload=True)

    if 'Fp1/F3' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'Fp1/F3':'Fp1' ,'Fp2/F4':'Fp2'})
        
    if '36' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'36':'Corr' ,'37':'Zygo','38':'Menton'})

    if 'E1' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'E1':'EOG1' ,'E2':'EOG2'})
    
    if 'Corru' in raw.info['ch_names']:
        raw.rename_channels({'Corru': 'Corr'})


    for ch in raw.info['ch_names']:
        if ch not in to_keep:
            raw.drop_channels([ch]) # only keeps channel that contains the trigger (where a stimulus was presented )

    raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})

    raw= raw.resample(sfreq=250)

    filter_params_emg = {'lpass': 100,'hpass': 10,'notches': [50]}
    raw.filter(l_freq=filter_params_emg['hpass'],h_freq=filter_params_emg['lpass'],picks=emg_ch)
    filter_params_eeg_eog = {'lpass': 70,'hpass': 0.3,'notches': [50]}
    raw.filter(l_freq=filter_params_eeg_eog['hpass'],h_freq=filter_params_eeg_eog['lpass'],picks=eeg_ch+eog_ch)
    raw.notch_filter(filter_params_eeg_eog['notches'], method='fft', picks=emg_ch+eeg_ch+eog_ch)

    frq=raw.info['sfreq']

    # create dataframe based on events 
    events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
    events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest (where a stimulus was presented)

    df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
    df_triggers.drop(columns=['dunno'],inplace=True)
    df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


    # incorporating meta_data 
    cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] # columns with relevant information from dataframe 
    trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc) 
    expected_muscle = ["Corr", "Zygo"]


 
    epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                    reject=None, preload=True, on_missing='warn')

   


    epochs.metadata = trial_info[(trial_info["Subject"] == subject) 
                                & (trial_info["Trigger"].isin([201.0, 202.0]))
                                & (trial_info["Nap_ID"] == int(block))]

    # adding metadata column for true activation
    # add another column for neither muscle being activated  
    true_activations = [] 
    for i in range(len(epochs.metadata)):
        true_ind = expected_muscle.index(epochs.metadata.iloc[i]["Expected_Muscle"]) 
        if (epochs.metadata.iloc[i]["Is_Correct"] == 0):
            if(epochs.metadata.iloc[i]["Nb_Corr"] < 3 and epochs.metadata.iloc[i]["Nb_Zygo"] < 3):
                true_activations.append("None")
            else:
                true_activations.append(expected_muscle[true_ind-1])
        else:
            true_activations.append(expected_muscle[true_ind])


    epochs.metadata["True_activation"] = true_activations
    return epochs, df_triggers

In [ ]:
def test_variance_window_func(feature):
    global window_test,baseline_window,threshold_test, window 

    events = np.zeros(len(feature),dtype=bool)

    baseline_mean_start = np.mean(feature[:baseline_window])
    baseline_mean_end = np.mean(feature[-baseline_window:-1])

    #----------NO RESPONSE CASE LOGIC--------------
    if ((abs(min(baseline_mean_start,baseline_mean_end) -
         np.max(feature))) < 250) or ((abs(min(baseline_mean_start,baseline_mean_end) -
         np.mean(feature))) < 10):
        return events, 0 
    
    #----------RESPONSE CASE LOGIC--------------
    max_var = 0 
    for start in range(0, len(feature) - window + 1, 1):
        stop = start + window
        feature_win = feature[start:stop]
        curr_var = np.mean(feature_win)

        if (curr_var >  max_var): 
            max_var = curr_var 
    

    thr =   max_var/4
    ind = np.where((feature> thr))
    events[ind] = True 

    
    return events,thr


In [6]:

''' 
def set_second_threshold(events):
    global r0 

    for i in range(len(events),len(events)-r0)
'''

' \ndef set_second_threshold(events):\n    global r0 \n\n    for i in range(len(events),len(events)-r0)\n'

### Testing 

In [ ]:
subject = 'RL13MT'
block = '04' #nap number
subject_epoch, _ = pre_process_subjets(subject,block)
current_index = 0 

In [ ]:
# extract epoch  
#current_index = 0 
epoch_zygo = np.squeeze(subject_epoch[current_index].get_data(picks=['Zygo']))
epoch_corr = np.squeeze(subject_epoch[current_index].get_data(picks=['Corr']))

# get features for epoch 
epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_fmd_zygo = get_features(epoch_zygo)
epoch_var_corr, epoch_rms_corr, epoch_wl_corr, epoch_fmd_corr = get_features(epoch_corr)

fig = plot_test_variance_window_func(epoch_corr,epoch_zygo,epoch_var_corr,epoch_var_zygo)

fig.canvas.mpl_connect('key_press_event', on_key)


title = fig.suptitle(
    f"subject: {subject} | block: {block} | epoch: {current_index}",
    fontsize=14,
)
plt.show()